# 03 · Preprocessing & modality merge

| | |
|---|---|
| **input**  | `data/formatted/<subject>/<session>.csv` |
| **output** | `data/clean/<subject>.csv` (one file per subject) |

Steps:
1. **EEG** — resample 2 kHz → 1 kHz, then remove non-brain ICA components    (ICLabel) and burst artefacts (ASR), fitted on quiet standing/sitting    segments.
2. **EMG** — rename the Japanese muscle-name headers to short romaji ids.
3. **Motion** — low-pass at 30 Hz, merge the X/Z ground-plane axes into    `*_XZ`, add 0.1 s / 0.2 s lagged copies.
4. Concatenate the three modalities on `t_sec`, keep the `label` column,    and stack all sessions of a subject with a `Session` id.

In [ ]:
import sys
from pathlib import Path

# make the `motion_intent` package importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_intent import config

In [ ]:
from motion_intent.preprocessing import (
    estimate_fs_from_time, decimate_to, clean_eeg,
    lowpass, merge_xz_all, drop_single_axis_marker_cols,
    extract_static_segments,
)

## Load a subject's sessions

In [ ]:
subject = 'subjectA'
session_paths = sorted((config.FORMATTED_DIR / subject).glob('*.csv'))
print(f'{len(session_paths)} sessions for {subject}')

## EEG: resample + artefact cleaning

In [ ]:
def preprocess_eeg(df):
    fs_in = estimate_fs_from_time(df['t_sec'])
    q = max(1, round(fs_in / config.FS))
    df = decimate_to(df, config.EEG_CH_ALL, q=q)

    calib = extract_static_segments(df, label_col='label', fs=config.FS)
    df = clean_eeg(df, config.EEG_CH_ALL, fs=config.FS, df_calib=calib)
    return df

## EMG: rename headers

In [ ]:
def preprocess_emg(df):
    return df.rename(columns=config.EMG_RENAME)

## Motion: low-pass, axis merge, lag features

In [ ]:
def preprocess_motion(df):
    raw_marker_cols = [c for c in df.columns if c.startswith('Markers_')]
    fs_m = estimate_fs_from_time(df['t_sec'])
    df = lowpass(df, raw_marker_cols, fs=fs_m, cutoff_hz=30.0, zero_phase=False)
    df = drop_single_axis_marker_cols(merge_xz_all(df))

    for lag_s in (0.1, 0.2):
        shift = int(lag_s * config.FS)
        for c in config.MOTION_COLS:
            df[f'{c}_t-{lag_s}s'] = df[c].shift(shift)
    return df

## Merge modalities and stack sessions

In [ ]:
frames = []
for i, path in enumerate(session_paths):
    df = pd.read_csv(path)
    df = preprocess_eeg(df)
    df = preprocess_emg(df)
    df = preprocess_motion(df)
    df['Session'] = i
    frames.append(df)

clean = pd.concat(frames, ignore_index=True)
out_path = config.CLEAN_DIR / f'{subject}.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(out_path, index=False)
print(out_path, clean.shape)